In [77]:
import warnings
warnings.filterwarnings('ignore')

In this notebook we would like to work with audiology dataset. 
we will:
1. Divide it to the sets: train, validiation and test. 
2. Extract 10 relevent meta-features. 
3. try some pipelines with Logistic Regression. 

let's begin

In [2]:
# Loading Audiology dataset 
import openml
import pandas as pd

# Load Audiology from TabZilla (OpenML Task 7)
task_id = 7

task = openml.tasks.get_task(task_id)
dataset = task.get_dataset()

# Load features and target (categorical_indicator - indicate which features are categoral)
X, y, categorical_indicator, attribute_names = dataset.get_data(
    target=task.target_name,
    dataset_format="dataframe"
)

print("Task ID:", task_id)
print("Dataset:", dataset.name)
print("X shape:", X.shape)
print("Target:", task.target_name)
print("Task type:", task.task_type)

display(X.head()) # X is a dataframe of Pandas which contains all the features of Audiology
display(y.value_counts()) # value_counts - how many samples are in each class

Task ID: 7
Dataset: audiology
X shape: (226, 69)
Target: class
Task type: Supervised Classification


,age_gt_60,air,airBoneGap,ar_c,ar_u,bone,boneAbnormal,bser,history_buzzing,history_dizziness,...,o_ar_u,s_sn_gt_1k,s_sn_gt_2k,s_sn_gt_4k,speech,static_normal,tymp,viith_nerve_signs,wave_V_delayed,waveform_ItoV_prolonged
0,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,good,t,a,f,f,f
1,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,good,t,a,f,f,f
2,t,normal,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,good,t,a,f,f,f
3,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,unmeasured,t,a,f,f,f
4,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,normal,t,a,f,f,f


class
cochlear_age                        57
cochlear_unknown                    48
cochlear_age_and_noise              22
normal_ear                          22
cochlear_poss_noise                 20
mixed_cochlear_unk_fixation          9
possible_menieres                    8
conductive_fixation                  6
otitis_media                         4
possible_brainstem_disorder          4
mixed_cochlear_age_otitis_media      4
mixed_cochlear_unk_ser_om            3
mixed_cochlear_age_s_om              2
mixed_cochlear_unk_discontinuity     2
conductive_discontinuity             2
cochlear_noise_and_heredity          2
retrocochlear_unknown                2
mixed_poss_noise_om                  2
mixed_cochlear_age_fixation          2
bells_palsy                          1
cochlear_age_plus_poss_menieres      1
acoustic_neuroma                     1
mixed_poss_central_om                1
poss_central                         1
Name: count, dtype: int64

We can see from the results above that our dataset is imbalanced. What problems can this cause?

1. A model that makes accurate predictions for the majority classes can achieve a high overall accuracy, even if it makes many mistakes on the minority classes.

2. We first split our data into training and test sets. Next, we take some samples from the training set to create a validation set. But what if we take the only sample belonging to a particular class? Then the model will not see that class during training.

In [3]:
#dividing Audiology to the sets train, validation and test

# We will use openML dividation of data to train, validiation and test
# train_idx - the indexses of the samples belongs to the train set
train_idx, test_idx = task.get_train_test_split_indices(
    repeat=0,
    fold=0, # A specific division of the sets train and test
    sample=0
)

#print(train_idx)

# iloc let us pick rows and colmuns by there index
X_train_full = X.iloc[train_idx] #pick all the rows that there indexes are in train_idx
X_test = X.iloc[test_idx]

y_train_full = y.iloc[train_idx]
y_test = y.iloc[test_idx]

#print(X_train_full)
#print(X.iloc[0]) # First sample (all its fetures values)


In [4]:
print("Available split dimensions:", task.get_split_dimensions())

print("Train indices:", train_idx[:10])
print("Test indices:", test_idx[:10])

print("Full dataset:", X.shape)
print("Train:", X_train_full.shape)
print("Test:", X_test.shape)

print("\nClass distribution in training data:")
display(y_train_full.value_counts())

Available split dimensions: (1, 10, 1)
Train indices: [ 89 194 212  50 193  75  19 167 204  76]
Test indices: [174  81  51  69 146 224 195 219 220 210]
Full dataset: (226, 69)
Train: (203, 69)
Test: (23, 69)

Class distribution in training data:


class
cochlear_age                        51
cochlear_unknown                    43
normal_ear                          20
cochlear_age_and_noise              19
cochlear_poss_noise                 18
mixed_cochlear_unk_fixation          8
possible_menieres                    7
conductive_fixation                  6
mixed_cochlear_age_otitis_media      4
possible_brainstem_disorder          4
otitis_media                         3
mixed_cochlear_unk_ser_om            3
mixed_cochlear_age_s_om              2
mixed_cochlear_unk_discontinuity     2
conductive_discontinuity             2
mixed_cochlear_age_fixation          2
retrocochlear_unknown                2
cochlear_noise_and_heredity          1
bells_palsy                          1
acoustic_neuroma                     1
cochlear_age_plus_poss_menieres      1
mixed_poss_noise_om                  1
mixed_poss_central_om                1
poss_central                         1
Name: count, dtype: int64

Becuase we have classes with only one sample (Class Imbalanced) we can't use "stratify" for validation set divition from train set. 

let's see the problem:

In [5]:
print("Number of classes:", y_train_full.nunique()) # how many diffrent values exist in y

print(
    "Classes with only one sample:",
    (y_train_full.value_counts() == 1).sum() 
)

print(
    "Smallest class size:",
    y_train_full.value_counts().min()
)

Number of classes: 24
Classes with only one sample: 7
Smallest class size: 1


What we can do for now ? we will keep all the lonely samples in the train. 

In [6]:
from sklearn.model_selection import train_test_split

# Identify classes with only one sample
class_counts = y_train_full.value_counts()
singleton_classes = class_counts[class_counts == 1].index

singleton_mask = y_train_full.isin(singleton_classes)

# Keep singleton classes in training
X_single = X_train_full.loc[singleton_mask]
y_single = y_train_full.loc[singleton_mask]

# Remaining samples
X_remaining = X_train_full.loc[~singleton_mask]
y_remaining = y_train_full.loc[~singleton_mask]

# Split remaining samples into Train and Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.2,
    random_state=42,
    stratify=y_remaining
)

# Add singleton samples back to training
X_train = pd.concat([X_train, X_single]) # Concatenate pandas objects along a particular axis
y_train = pd.concat([y_train, y_single])

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

display(X_train.dtypes.value_counts())
display(y_train.value_counts())

Train: (163, 69)
Validation: (40, 69)
Test: (23, 69)


category    60
category     4
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64

class
cochlear_age                        41
cochlear_unknown                    34
normal_ear                          16
cochlear_age_and_noise              15
cochlear_poss_noise                 14
mixed_cochlear_unk_fixation          6
conductive_fixation                  5
possible_menieres                    5
mixed_cochlear_age_otitis_media      3
possible_brainstem_disorder          3
mixed_cochlear_age_s_om              2
otitis_media                         2
mixed_cochlear_unk_ser_om            2
mixed_cochlear_unk_discontinuity     2
mixed_cochlear_age_fixation          2
conductive_discontinuity             2
retrocochlear_unknown                2
cochlear_noise_and_heredity          1
bells_palsy                          1
acoustic_neuroma                     1
cochlear_age_plus_poss_menieres      1
mixed_poss_noise_om                  1
mixed_poss_central_om                1
poss_central                         1
Name: count, dtype: int64

In [7]:
print("Train:")
display(X_train.head())

print("Validation:")
display(X_val.head())

print("Test:")
display(X_test.head())

Train:


,age_gt_60,air,airBoneGap,ar_c,ar_u,bone,boneAbnormal,bser,history_buzzing,history_dizziness,...,o_ar_u,s_sn_gt_1k,s_sn_gt_2k,s_sn_gt_4k,speech,static_normal,tymp,viith_nerve_signs,wave_V_delayed,waveform_ItoV_prolonged
37,t,mild,f,normal,normal,NaN,t,NaN,f,f,...,normal,f,f,f,good,t,a,f,f,f
196,t,mild,f,normal,elevated,mild,t,NaN,f,t,...,normal,f,f,f,good,t,a,f,f,f
14,t,normal,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,very_good,t,a,f,f,f
9,f,normal,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,very_good,t,a,f,f,f
189,t,mild,f,normal,elevated,unmeasured,t,NaN,f,f,...,normal,f,f,f,unmeasured,t,a,f,f,f


Validation:


,age_gt_60,air,airBoneGap,ar_c,ar_u,bone,boneAbnormal,bser,history_buzzing,history_dizziness,...,o_ar_u,s_sn_gt_1k,s_sn_gt_2k,s_sn_gt_4k,speech,static_normal,tymp,viith_nerve_signs,wave_V_delayed,waveform_ItoV_prolonged
223,f,normal,f,normal,normal,unmeasured,f,degraded,f,f,...,normal,f,f,f,normal,f,a,f,f,f
163,f,mild,f,normal,normal,NaN,t,NaN,f,f,...,normal,f,f,f,good,t,a,f,f,f
123,t,moderate,f,elevated,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,very_poor,t,a,f,f,f
182,f,normal,f,normal,normal,normal,f,NaN,f,f,...,normal,f,f,f,normal,t,a,f,f,f
179,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,poor,t,a,f,f,f


Test:


,age_gt_60,air,airBoneGap,ar_c,ar_u,bone,boneAbnormal,bser,history_buzzing,history_dizziness,...,o_ar_u,s_sn_gt_1k,s_sn_gt_2k,s_sn_gt_4k,speech,static_normal,tymp,viith_nerve_signs,wave_V_delayed,waveform_ItoV_prolonged
174,f,mild,f,normal,normal,mild,f,NaN,f,f,...,normal,f,f,f,normal,t,a,f,f,f
81,f,normal,f,elevated,elevated,normal,f,NaN,f,f,...,absent,f,f,f,normal,t,a,f,f,f
51,f,normal,f,normal,normal,normal,t,NaN,f,f,...,elevated,f,f,f,very_good,t,a,f,f,f
69,f,mild,f,elevated,normal,NaN,f,NaN,f,f,...,elevated,f,f,f,normal,t,a,f,f,f
146,f,profound,f,NaN,NaN,NaN,t,NaN,f,f,...,NaN,f,f,f,NaN,t,a,f,f,f


## Extarctig Meta-Features

Now we will create a function called *def profile_dataset(X_train, y_train)* which extracts some relevent meta-features from Audiology dataset

In [8]:
def profile_dataset(X_train, y_train):

    #number of samples and features
    n_samples, n_features = X_train.shape #Tuple unpacking

    # P/N ratio
    features_to_sample_ratio = n_features/n_samples

    # Categorical features
    # display(X_train.dtypes) #returns the data type of every column so we can see which ones are categorial
    categorical_features = X_train.select_dtypes(
    include=["category", "object"]
    )
    n_categorical = categorical_features.shape[1]

    # Numerical features
    numerical_features = X_train.select_dtypes(
        include=["number"]
    )
    n_numerical = numerical_features.shape[1]

    # missing ratio - number of missing cells divided by the overall number of cells
    #display(X_train.isna()) # return a matrix same size of X_train but with True for missing values and False for existing values
    n_missing = X_train.isna().sum().sum() # True is equivalent to 1 for every column and another sum to go over all the table
    missing_ratio = n_missing / (n_samples*n_features) # or n_missing /(X_train.size)

    # class distrabution
    #display(y_train.value_counts()) # returns how many samples each class has
    class_distribution = y_train.value_counts( # relative frequency of each class
    normalize=True
    ).to_dict()
    #display(y_train.value_counts(normalize=True))

    # categorical cardinality
    #display(X_train.nunique()) # returns unique values in every column (for example:["Red", "Blue", "Red", "Green", "Blue"] => ["Red", "Blue", "Green"] =>  cardinality = 3 ) 
    categorical_cardinality= categorical_features.nunique().to_dict()
    #display(categorical_features.nunique())

    profile = {
        "n_samples: " : n_samples,
        "n_features: " : n_features,
        "features to sample ratio (P/N): ": features_to_sample_ratio,
        "Categorical features: ": n_categorical, 
        "numerical features": n_numerical,
        "number of missing cells: " : n_missing,
        "missing ratio: ": missing_ratio,
        "class_distribution": class_distribution,
        "categorical_cardinality": categorical_cardinality

    }

    return profile

    
profile = profile_dataset(X_train, y_train)

print("\n")
display(pd.DataFrame(
    profile.items(),
    columns=["Meta-feature","Value"]
))



,Meta-feature,Value
0,n_samples:,163
1,n_features:,69
2,features to sample ratio (P/N):,0.423313
3,Categorical features:,69
4,numerical features,0
5,number of missing cells:,229
6,missing ratio:,0.020361
7,class_distribution,"{'cochlear_age': 0.25153374233128833, 'cochlea..."
8,categorical_cardinality,"{'age_gt_60': 2, 'air': 4, 'airBoneGap': 2, 'a..."


## Baseline Pipeline — Logistic Regression

Pipeline: Imputation → One-Hot Encoding → Logistic Regression.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer # for imputaion
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from time import perf_counter

In [ ]:
# Preprocessing
categorical_columns = X_train.select_dtypes(
    include=["category","object"]
).columns.tolist()

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # if a category is missing we will complete it with the most frequent category in the feature
    ("encoder", OneHotEncoder(handle_unknown="ignore")) 
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, categorical_columns)
    ]
)

In [20]:
# With LR
baseline_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    ))
])

display(baseline_pipeline)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['age_gt_60', 'air',
                                                   'airBoneGap', 'ar_c', 'ar_u',
                                                   'bone', 'boneAbnormal',
                                                   'bser', 'history_buzzing',
                                                   'history_dizziness',
                                                   'history_fluctuating',
                                                   'history_fulln...
                                                   'history_heredity',
                                                   'history_nausea',
                                                   'history_noise',
                                                   'history_recruitment',
                                                   'history_ringing',
                                                   'history_roaring',
                                                   'history_vomiting',
                                                   'late_wave_poor', 'm_at_2k',
                                                   'm_cond_lt_1k', 'm_gt_1k',
                                                   'm_m_gt_2k', 'm_m_sn',
                                                   'm_m_sn_gt_1k',
                                                   'm_m_sn_gt_2k',
                                                   'm_m_sn_gt_500',
                                                   'm_p_sn_gt_2k', 'm_s_gt_500', ...])])),
                ('classifier',
                 LogisticRegression(max_iter=2000, random_state=42))])

In [22]:
# Training the complete pipeline
start = perf_counter()

baseline_pipeline.fit(X_train, y_train) # automaticlly activate all the pipeline steps

train_time = perf_counter() - start

# Predict validation labels
y_preb = baseline_pipeline.predict(X_val) # using what we learned about the training set to predit on the validation set

# Evaluation
accuracy = accuracy_score(y_val, y_preb)
balanced_accuracy = balanced_accuracy_score(y_val, y_preb)

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"Training Time: {train_time:.3f} seconds")


Validation Accuracy: 0.9000
Validation Balanced Accuracy: 0.7833
Training Time: 0.081 seconds


## Scaled Pipeline — Logistic Regression

Pipeline: Imputation → One-Hot Encoding → Scallnig → Logistic Regression.

In [41]:
from sklearn.preprocessing import StandardScaler

In [74]:
# Select categorical and numerical columns
categorical_columns = X_train.select_dtypes(
    include=["category","object"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Mandatory preprocessing
categorical_pipeline_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # if a category is missing we will complete it with the most frequent category in the feature
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Numerical preprocessing
numerical_pipeline_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # if a category is missing we will complete it with the most frequent category in the feature
    ("scaler", StandardScaler())
])

# Combine preprocessing steps
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline_scaled, categorical_columns),
        ("numerical", numerical_pipeline_scaled, numerical_columns)
    ]
)

In [75]:
# With LR
scaled_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor_scaled),
    ("scaler", StandardScaler(
        with_mean=False)), 
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    ))
])

display(scaled_pipeline)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['age_gt_60', 'air',
                                                   'airBoneGap', 'ar_c', 'ar_u',
                                                   'bone', 'boneAbnormal',
                                                   'bser', 'history_buzzing',
                                                   'history_dizziness',
                                                   'history_fluctuating',
                                                   'history_fulln...
                                                   'm_cond_lt_1k', 'm_gt_1k',
                                                   'm_m_gt_2k', 'm_m_sn',
                                                   'm_m_sn_gt_1k',
                                                   'm_m_sn_gt_2k',
                                                   'm_m_sn_gt_500',
                                                   'm_p_sn_gt_2k', 'm_s_gt_500', ...]),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  [])])),
                ('scaler', StandardScaler(with_mean=False)),
                ('classifier',
                 LogisticRegression(max_iter=2000, random_state=42))])

In [78]:
# Training the complete pipeline
start = perf_counter()

scaled_pipeline.fit(X_train, y_train) # automaticlly activate all the pipeline steps

train_time_scaled = perf_counter() - start

# Predict validation labels
y_preb_scaled = scaled_pipeline.predict(X_val) # using what we learned about the training set to predit on the validation set

# Evaluation
accuracy_scaled = accuracy_score(y_val, y_preb_scaled)
balanced_accuracy_scaled = balanced_accuracy_score(y_val, y_preb_scaled)

print(f"Validation Accuracy: {accuracy_scaled:.4f}")
print(f"Validation Balanced Accuracy: {balanced_accuracy_scaled:.4f}")
print(f"Training Time: {train_time_scaled:.3f} seconds")


Validation Accuracy: 0.8500
Validation Balanced Accuracy: 0.7208
Training Time: 0.150 seconds


## Feature selection pipeline — Logistic Regression

Pipeline: Imputation → One-Hot Encoding → feature selection → Logistic Regression.

In [64]:
from sklearn.feature_selection import SelectKBest, f_classif #  removes all but the k highest scoring features

In [68]:
# Select categorical and numerical columns
categorical_columns = X_train.select_dtypes(
    include=["category","object"]
).columns.tolist()

# Mandatory preprocessing
categorical_pipeline_fs = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# Combine preprocessing steps
preprocessor_fs = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline_fs, categorical_columns)
    ]
)

In [69]:
# With LR
fs_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor_fs),
    ("feature selection", SelectKBest(
        score_func=f_classif,
        k=30
    )),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    ))
])

display(fs_pipeline)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['age_gt_60', 'air',
                                                   'airBoneGap', 'ar_c', 'ar_u',
                                                   'bone', 'boneAbnormal',
                                                   'bser', 'history_buzzing',
                                                   'history_dizziness',
                                                   'history_fluctuating',
                                                   'history_fulln...
                                                   'history_noise',
                                                   'history_recruitment',
                                                   'history_ringing',
                                                   'history_roaring',
                                                   'history_vomiting',
                                                   'late_wave_poor', 'm_at_2k',
                                                   'm_cond_lt_1k', 'm_gt_1k',
                                                   'm_m_gt_2k', 'm_m_sn',
                                                   'm_m_sn_gt_1k',
                                                   'm_m_sn_gt_2k',
                                                   'm_m_sn_gt_500',
                                                   'm_p_sn_gt_2k', 'm_s_gt_500', ...])])),
                ('feature selection', SelectKBest(k=30)),
                ('classifier',
                 LogisticRegression(max_iter=2000, random_state=42))])

In [79]:
# Training the complete pipeline
start = perf_counter()

fs_pipeline.fit(X_train, y_train) # automaticlly activate all the pipeline steps

train_time_fs = perf_counter() - start

# Predict validation labels
y_preb_fs = fs_pipeline.predict(X_val) # using what we learned about the training set to predit on the validation set

# Evaluation
accuracy_fs = accuracy_score(y_val, y_preb_fs)
balanced_accuracy_fs = balanced_accuracy_score(y_val, y_preb_fs)

print(f"Validation Accuracy: {accuracy_fs:.4f}")
print(f"Validation Balanced Accuracy: {balanced_accuracy_fs:.4f}")
print(f"Training Time: {train_time_fs:.3f} seconds")


Validation Accuracy: 0.7250
Validation Balanced Accuracy: 0.5241
Training Time: 0.034 seconds


In [80]:
results = pd.DataFrame([
    {
        "Pipeline": "Baseline",
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_accuracy,
        "Train Time (s)": train_time
    },
    {
        "Pipeline": "With Scaling",
        "Accuracy": accuracy_scaled,
        "Balanced Accuracy": balanced_accuracy_scaled,
        "Train Time (s)": train_time_scaled
    },
    {
        "Pipeline": "With Feature Selection",
                "Accuracy": accuracy_fs,
                "Balanced Accuracy": balanced_accuracy_fs,
                "Train Time (s)": train_time_fs
    }
])

display(results)

,Pipeline,Accuracy,Balanced Accuracy,Train Time (s)
0,Baseline,0.900,0.783333,0.180920
1,With Scaling,0.850,0.720833,0.149748
2,With Feature Selection,0.725,0.524074,0.034487
